# Milestone 01: Sequence Foundations
## Paper 00: Jeffrey L. Elman (1990) — *"Finding Structure in Time"*

> **"How can a fixed-size neural network represent time without spatializing it?"**  
> — *Jeffrey L. Elman, Cognitive Science (1990)*

---

### Learning Goals of This Lab:
1. Understand the **Simple Recurrent Network (SRN)** architecture from first principles.
2. Implement the **forward recurrence loop** in pure NumPy.
3. Manually derive and code **Backpropagation Through Time (BPTT)** using the multivariate chain rule.
4. Mathematically verify analytical derivatives using finite-difference numerical gradient checking ($< 10^{-5}$).
5. Numerically plot and prove the **Vanishing Gradient Problem** that led to the invention of LSTMs.
6. Train an Elman RNN on a character-level sequence to observe emergent temporal structure.


### 1. Architecture & The Recurrent Feedback Equation

$$\mathbf{a}_t = \mathbf{x}_t \mathbf{W}_{xh} + \mathbf{h}_{t-1} \mathbf{W}_{hh} + \mathbf{b}_h$$
$$\mathbf{h}_t = \tanh(\mathbf{a}_t)$$
$$\mathbf{z}_t = \mathbf{h}_t \mathbf{W}_{hy} + \mathbf{b}_y$$
$$\hat{\mathbf{y}}_t = \text{softmax}(\mathbf{z}_t)$$

Where:
* $\mathbf{x}_t \in \mathbb{R}^{B \times d_x}$: Current sensory input at time $t$
* $\mathbf{h}_t \in \mathbb{R}^{B \times d_h}$: Recurrent hidden state (compressed memory of past)
* $\mathbf{W}_{hh} \in \mathbb{R}^{d_h \times d_h}$: Internal feedback matrix (the self-loop)


In [1]:
import numpy as np
import matplotlib.pyplot as plt

print("NumPy & Matplotlib loaded successfully!")

NumPy & Matplotlib loaded successfully!


### 2. The Elman RNN Class Implementation

In [ ]:
class ElmanRNN:
    def __init__(self, input_dim, hidden_dim, output_dim, seed=42):
        np.random.seed(seed)
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # Xavier / Glorot Initialization
        self.W_xh = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / (input_dim + hidden_dim))
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * np.sqrt(2.0 / (hidden_dim + hidden_dim))
        self.b_h  = np.zeros((1, hidden_dim))
        
        self.W_hy = np.random.randn(hidden_dim, output_dim) * np.sqrt(2.0 / (hidden_dim + output_dim))
        self.b_y  = np.zeros((1, output_dim))
        
    @staticmethod
    def softmax(z):
        shift_z = z - np.max(z, axis=-1, keepdims=True)
        exp_z = np.exp(shift_z)
        return exp_z / np.sum(exp_z, axis=-1, keepdims=True)
        
    def forward(self, X, h_init=None):
        T, B, _ = X.shape
        if h_init is None:
            h_init = np.zeros((B, self.hidden_dim))
            
        h_states = [h_init]
        y_probs = []
        
        for t in range(T):
            x_t = X[t]
            h_prev = h_states[-1]
            a_t = np.dot(x_t, self.W_xh) + np.dot(h_prev, self.W_hh) + self.b_h
            h_t = np.tanh(a_t)
            z_t = np.dot(h_t, self.W_hy) + self.b_y
            p_t = self.softmax(z_t)
            
            h_states.append(h_t)
            y_probs.append(p_t)
            
        return y_probs, h_states
        
    def backward(self, X, Y, y_probs, h_states):
        T, B, _ = X.shape
        dW_xh = np.zeros_like(self.W_xh)
        dW_hh = np.zeros_like(self.W_hh)
        db_h  = np.zeros_like(self.b_h)
        dW_hy = np.zeros_like(self.W_hy)
        db_y  = np.zeros_like(self.b_y)
        
        dh_next = np.zeros((B, self.hidden_dim))
        dh_norms = []
        
        for t in reversed(range(T)):
            dz_t = y_probs[t] - Y[t]
            dW_hy += np.dot(h_states[t + 1].T, dz_t)
            db_y  += np.sum(dz_t, axis=0, keepdims=True)
            
            dh_t = np.dot(dz_t, self.W_hy.T) + dh_next
            dh_norms.append(np.linalg.norm(dh_t))
            
            da_t = dh_t * (1.0 - h_states[t + 1] ** 2)
            dW_xh += np.dot(X[t].T, da_t)
            dW_hh += np.dot(h_states[t].T, da_t)
            db_h  += np.sum(da_t, axis=0, keepdims=True)
            
            dh_next = np.dot(da_t, self.W_hh.T)
            
        return {
            "dW_xh": dW_xh / B, "dW_hh": dW_hh / B, "db_h": db_h / B,
            "dW_hy": dW_hy / B, "db_y": db_y / B,
            "dh_norms": list(reversed(dh_norms))
        }
        
    def compute_loss(self, y_probs, Y, eps=1e-12):
        T, B, _ = Y.shape
        loss = 0.0
        for t in range(T):
            p_t = np.clip(y_probs[t], eps, 1.0 - eps)
            loss -= np.sum(Y[t] * np.log(p_t))
        return loss / (T * B)

print("ElmanRNN Class compiled successfully!")

### 3. Analytical GradCheck (Finite Differences)
We prove that our manual BPTT analytical gradient matches the true numerical derivative within $< 10^{-5}$:

In [ ]:
T, B, Din, Dhidden, Dout = 4, 2, 3, 5, 4
X = np.random.randn(T, B, Din)
targets_idx = np.random.randint(0, Dout, size=(T, B))
Y = np.zeros((T, B, Dout))
for t in range(T):
    for b in range(B):
        Y[t, b, targets_idx[t, b]] = 1.0

model = ElmanRNN(Din, Dhidden, Dout, seed=123)
probs, states = model.forward(X)
analytical_grads = model.backward(X, Y, probs, states)

eps = 1e-5
for param_name in ["W_hh", "W_xh", "W_hy"]:
    param = getattr(model, param_name)
    grad_analytical = analytical_grads["d" + param_name]
    grad_numerical = np.zeros_like(param)
    it = np.nditer(param, flags=["multi_index"], op_flags=["readwrite"])
    while not it.finished:
        ix = it.multi_index
        old_val = param[ix]
        param[ix] = old_val + eps
        p_pos, _ = model.forward(X)
        l_pos = model.compute_loss(p_pos, Y)
        param[ix] = old_val - eps
        p_neg, _ = model.forward(X)
        l_neg = model.compute_loss(p_neg, Y)
        param[ix] = old_val
        grad_numerical[ix] = (l_pos - l_neg) / (2.0 * eps) * T
        it.iternext()
    rel_err = np.linalg.norm(grad_analytical - grad_numerical) / (np.linalg.norm(grad_analytical) + np.linalg.norm(grad_numerical) + 1e-12)
    print(f"{param_name}: Relative Error = {rel_err:.2e} -> {'PASSED' if rel_err < 1e-5 else 'FAILED'}")

### 4. The Vanishing Gradient Problem (The Bridge to LSTM)
Let us observe how the gradient norm $\|\frac{\partial \mathcal{L}}{\partial \mathbf{h}_t}\|$ decays exponentially as it travels back through time.

In [ ]:
T_long = 35
X_long = np.random.randn(T_long, 1, 8)
Y_long = np.zeros((T_long, 1, 8))
Y_long[-1, 0, 0] = 1.0  # Loss evaluated at final step

model_long = ElmanRNN(8, 16, 8, seed=42)
probs, states = model_long.forward(X_long)
grads = model_long.backward(X_long, Y_long, probs, states)
dh_norms = grads["dh_norms"]

plt.figure(figsize=(10, 4), dpi=120)
plt.plot(range(T_long), dh_norms, marker="o", color="#e74c3c", lw=2, label="||dh_t / dLoss||")
plt.title("The Vanishing Gradient Problem in Elman RNN (BPTT)", fontsize=14, fontweight="bold")
plt.xlabel("Time-step t (0 = furthest in past, 34 = present)", fontsize=12)
plt.ylabel("Gradient Norm", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.show()